# Profit evidence

Inspect the append-only paper ledger. Probability quality remains the first gate; ROI, CLV, drawdown, and results by league/market/edge bucket determine whether an apparent edge has survived real quoted prices.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
DB = ROOT / 'data/state/oracle_bets.db'
connection = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection)
display(tables)

In [ ]:
positions = pd.read_sql_query('SELECT * FROM paper_positions', connection)
settlements = pd.read_sql_query('SELECT * FROM settlements', connection)
print(f"Paper positions: {len(positions):,} | settlements: {len(settlements):,}")
display(positions.tail(20))
display(settlements.tail(20))

In [ ]:
if settlements.empty:
    print('No settled paper bets yet; do not interpret theoretical edges as profit evidence.')
else:
    numeric = settlements.apply(pd.to_numeric, errors='ignore')
    display(numeric.describe(include='all').T)
    pnl_column = next((col for col in ('pnl_units', 'profit_units', 'pnl') if col in numeric), None)
    if pnl_column:
        numeric[pnl_column].cumsum().plot(title='Cumulative paper P&L', ylabel='units')
connection.close()